In [ ]:
# Uji Gerak Wan 2.2 TI2V-5B - CAHAYA TERAKHIR (S6, S18, S22)
# Batch kernel: install deps, decode stills, render 3 clips to /kaggle/working
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'diffusers', 'transformers', 'accelerate', 'imageio', 'imageio-ffmpeg', 'sentencepiece'], check=False)
print('deps ok')


In [ ]:
import base64, pathlib
S = pathlib.Path('/kaggle/working/stills'); S.mkdir(parents=True, exist_ok=True)
DATA = {
    's06': '...',
    's18': '...',
    's22': '...',
}
for k, v in DATA.items():
    (S / f'{k}.webp').write_bytes(base64.b64decode(v))
from PIL import Image
for k in DATA:
    Image.open(S / f'{k}.webp').convert('RGB').save(S / f'{k}.png')
print('stills ready:', sorted(p.name for p in S.iterdir()))


In [ ]:

import gc, os, time, traceback
import torch
from PIL import Image
from diffusers.utils import export_to_video

NEG = 'no warped face, no face change, no outfit change, no extra fingers, no extra limbs, no text, no watermark, no cartoon look, no flicker'
SHOTS = [
    ('s06', 'A young Indonesian girl with a red ikat headband and high ponytail, close-up portrait. Slow gentle push-in. Strands of her hair and her ponytail sway softly in a light wind, she blinks slowly and her gaze lifts with quiet determination, thin mist drifts behind her, warm golden lantern light flickers faintly from below the frame. Cinematic photorealistic fantasy, soft golden-hour light, shallow depth of field, gentle film grain, teal-and-amber grade, smooth cinematic motion.'),
    ('s18', 'Extreme close-up of two hands gently cupping a small antique clay lantern. The tiny flame grows steadily into a bright warm blaze, concentric rings of golden light expand outward in slow pulses, golden sparks rise and swirl upward, gray mist recoils and dissolves at the edges of the frame. The hands remain still. Cinematic photorealistic fantasy, gentle film grain, teal-and-amber grade, smooth cinematic motion.'),
    ('s22', 'High aerial drone shot slowly pushing forward over a vast rolling savanna valley and winding river. A living wave of vivid color — emerald green, gold and warm red — sweeps across the land from foreground to background, dissolving the gray mist wherever it passes, the river sparkling as color returns. Cinematic photorealistic fantasy, gentle film grain, smooth cinematic motion.'),
]

MODEL = 'Wan-AI/Wan2.2-TI2V-5B-Diffusers'
try:
    from diffusers import WanPipeline as PipeCls
except ImportError:
    from diffusers import WanImageToVideoPipeline as PipeCls
print('pipeline class:', PipeCls.__name__)
pipe = PipeCls.from_pretrained(MODEL, torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()
try:
    pipe.vae.enable_tiling()
except Exception:
    pass

for key, prompt in SHOTS:
    t0 = time.time()
    try:
        img = Image.open(f'/kaggle/working/stills/{key}.png').convert('RGB')
        out = pipe(image=img, prompt=prompt, negative_prompt=NEG,
                   height=1280, width=720, num_frames=49,
                   num_inference_steps=20, guidance_scale=5.0).frames[0]
        path = f'/kaggle/working/uji-{key}.mp4'
        export_to_video(out, path, fps=16)
        print(f'DONE {key}: {os.path.getsize(path)} bytes in {time.time()-t0:.0f}s', flush=True)
    except Exception:
        print(f'FAILED {key} after {time.time()-t0:.0f}s')
        traceback.print_exc()
    gc.collect(); torch.cuda.empty_cache()
print('ALL FINISHED')


In [ ]:
import os
for f in sorted(os.listdir('/kaggle/working')):
    if f.endswith('.mp4'):
        print(f, os.path.getsize('/kaggle/working/' + f))
